# STARE Hybrid V6 — End-to-End Hybrid TransUNet + Feature-GNN

Same core Hybrid V6 architecture as the DRIVE experiment:
**CNN + multi-scale edge enhancement → parallel Transformer/feature-GNN → learned fusion → attention-gated decoder**.

- No pretrained model
- NO LOO
- Fixed 16/4 STARE split
- GNN operates on learned bottleneck embeddings and is trained end-to-end
- Multi-scale dilations 1/2/3 at early encoder stages
- Four GNN message-passing layers
- Attention-gated skip connections
- Deep supervision
- Loss: 50% Dice + 30% Focal-Tversky + 20% clDice
- Validation threshold optimization, TTA, FOV-aware metrics
- All 20 STARE prediction visuals


## 1. Upload and extract STARE


In [ ]:
from google.colab import files
import os, tarfile, gzip, glob, shutil
from pathlib import Path

print("Upload BOTH files:")
print("  1) stare-images.tar (or stare-images(1).tar)")
print("  2) labels-ah.tar   (or labels-ah(1).tar)")
uploaded = files.upload()

STARE_ROOT="/content/stare_data"
IMG_DIR=os.path.join(STARE_ROOT,"images")
GT_DIR=os.path.join(STARE_ROOT,"labels-ah")
os.makedirs(IMG_DIR,exist_ok=True)
os.makedirs(GT_DIR,exist_ok=True)

for name in uploaded:
    src="/content/"+name
    target=GT_DIR if "label" in name.lower() or "ah" in name.lower() else IMG_DIR
    if tarfile.is_tarfile(src):
        with tarfile.open(src,"r:*") as t:
            t.extractall(target)

# STARE archives contain .ppm.gz files. Decompress them.
for root in [IMG_DIR,GT_DIR]:
    for gz in glob.glob(root+"/**/*.gz",recursive=True):
        dst=gz[:-3]
        if not os.path.exists(dst):
            with gzip.open(gz,"rb") as fi, open(dst,"wb") as fo:
                shutil.copyfileobj(fi,fo)

image_files=sorted(glob.glob(IMG_DIR+"/**/*.ppm",recursive=True))
label_files=sorted(glob.glob(GT_DIR+"/**/*.ppm",recursive=True))

print("STARE images found:",len(image_files))
print("AH labels found   :",len(label_files))
assert len(image_files)==20, f"Expected 20 STARE images, found {len(image_files)}"
assert len(label_files)==20, f"Expected 20 AH labels, found {len(label_files)}"
print("Dataset ready:",STARE_ROOT)


## 2. Install dependencies


In [ ]:
!pip install -q pillow numpy torch torchvision scikit-image scikit-learn scipy matplotlib opencv-python-headless pandas


## 3. Imports, preprocessing, augmentation and datasets


In [ ]:
import os, glob, random, math, csv
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image

from skimage.morphology import skeletonize, dilation, disk
from skimage.metrics import structural_similarity
from scipy.ndimage import distance_transform_edt

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    confusion_matrix
)

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def enhance_retina_rgb(img_pil):
    arr = np.array(img_pil.convert("RGB"))
    g = arr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g2 = clahe.apply(g)

    out = arr.astype(np.float32)
    out[:, :, 1] = 0.35 * out[:, :, 1] + 0.65 * g2
    out[:, :, 0] = 0.80 * out[:, :, 0] + 0.20 * g2
    out[:, :, 2] = 0.80 * out[:, :, 2] + 0.20 * g2

    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))

def paired_augment(img, mask):
    if torch.rand(1).item() < 0.5:
        img, mask = TF.hflip(img), TF.hflip(mask)
    if torch.rand(1).item() < 0.5:
        img, mask = TF.vflip(img), TF.vflip(mask)

    angle = float(torch.empty(1).uniform_(-30, 30))
    scale = float(torch.empty(1).uniform_(0.90, 1.10))
    shear = [float(torch.empty(1).uniform_(-8, 8)), 0.0]

    img = TF.affine(
        img, angle=angle, translate=[0,0], scale=scale, shear=shear,
        interpolation=InterpolationMode.BILINEAR
    )
    mask = TF.affine(
        mask, angle=angle, translate=[0,0], scale=scale, shear=shear,
        interpolation=InterpolationMode.NEAREST
    )

    if torch.rand(1).item() < 0.7:
        img = TF.adjust_brightness(img, float(torch.empty(1).uniform_(0.80, 1.20)))
        img = TF.adjust_contrast(img, float(torch.empty(1).uniform_(0.80, 1.25)))

    if torch.rand(1).item() < 0.5:
        img = TF.adjust_gamma(img.clamp(0,1), float(torch.empty(1).uniform_(0.75, 1.35)))

    return img, (mask > 0.5).float()

class StareDataset(Dataset):
    def __init__(self, root_dir, image_size=(512,512), augment=False):
        self.image_size=image_size
        self.augment=augment
        self.images=sorted(glob.glob(os.path.join(root_dir,"images","**","*.ppm"),recursive=True))
        self.masks=sorted(glob.glob(os.path.join(root_dir,"labels-ah","**","*.ppm"),recursive=True))
        assert len(self.images)==len(self.masks)==20

    def __len__(self): return 20

    def __getitem__(self,idx):
        img=Image.open(self.images[idx]).convert("RGB").resize(self.image_size,Image.BILINEAR)
        mask=Image.open(self.masks[idx]).convert("L").resize(self.image_size,Image.NEAREST)

        # STARE AH manual label: bright pixels are vessel.
        mask_np=(np.array(mask)>20).astype(np.float32)

        # Estimate retinal field of view from the original fundus.
        arr=np.array(img)
        gray=cv2.cvtColor(arr,cv2.COLOR_RGB2GRAY)
        fov=(gray>10).astype(np.uint8)
        fov=cv2.morphologyEx(fov,cv2.MORPH_CLOSE,np.ones((15,15),np.uint8))
        n,lab,stats,_=cv2.connectedComponentsWithStats(fov,8)
        if n>1:
            k=1+np.argmax(stats[1:,cv2.CC_STAT_AREA])
            fov=(lab==k).astype(np.float32)
        else:
            fov=fov.astype(np.float32)

        img=enhance_retina_rgb(img)
        img_t=TF.to_tensor(img)
        mask_t=torch.from_numpy(mask_np).unsqueeze(0)
        fov_t=torch.from_numpy(fov).unsqueeze(0)

        if self.augment:
            # Apply identical geometry to image/mask/FOV.
            if torch.rand(1).item()<0.5:
                img_t,mask_t,fov_t=TF.hflip(img_t),TF.hflip(mask_t),TF.hflip(fov_t)
            if torch.rand(1).item()<0.5:
                img_t,mask_t,fov_t=TF.vflip(img_t),TF.vflip(mask_t),TF.vflip(fov_t)
            angle=float(torch.empty(1).uniform_(-30,30))
            scale=float(torch.empty(1).uniform_(0.90,1.10))
            img_t=TF.affine(img_t,angle,[0,0],scale,[0.,0.],InterpolationMode.BILINEAR)
            mask_t=TF.affine(mask_t,angle,[0,0],scale,[0.,0.],InterpolationMode.NEAREST)
            fov_t=TF.affine(fov_t,angle,[0,0],scale,[0.,0.],InterpolationMode.NEAREST)
            if torch.rand(1).item()<0.7:
                img_t=TF.adjust_brightness(img_t,float(torch.empty(1).uniform_(0.80,1.20)))
                img_t=TF.adjust_contrast(img_t,float(torch.empty(1).uniform_(0.80,1.25)))

        img_t=TF.normalize(img_t,[0.5]*3,[0.5]*3)
        return img_t,(mask_t>0.5).float(),(fov_t>0.5).float()

class ThinVesselPatchDataset(Dataset):
    def __init__(self,base_subset,patch_size=256,patches_per_image=32,min_vessel_fraction=0.006,max_vessel_fraction=0.12):
        self.base=base_subset; self.patch=patch_size; self.ppi=patches_per_image
        self.min_v=min_vessel_fraction; self.max_v=max_vessel_fraction
    def __len__(self): return len(self.base)*self.ppi
    def __getitem__(self,idx):
        img,mask,fov=self.base[idx//self.ppi]
        _,H,W=img.shape; p=self.patch
        vessel=torch.nonzero(mask[0]>0.5,as_tuple=False)
        best=None
        for _ in range(24):
            # Prefer patches centered near vessels to improve thin-vessel learning.
            if len(vessel)>0 and random.random()<0.65:
                r,c=vessel[random.randrange(len(vessel))].tolist()
                top=max(0,min(H-p,r-random.randint(p//4,3*p//4)))
                left=max(0,min(W-p,c-random.randint(p//4,3*p//4)))
            else:
                top=random.randint(0,H-p); left=random.randint(0,W-p)
            x=img[:,top:top+p,left:left+p]
            y=mask[:,top:top+p,left:left+p]
            fv=fov[:,top:top+p,left:left+p]
            frac=float((y*fv).sum()/(fv.sum()+1e-6))
            best=(x,y,fv)
            if self.min_v<=frac<=self.max_v: break
        return best


## 4. Hybrid V6 architecture — CNN + multi-scale edge enhancement → Transformer + feature-GNN → learned fusion → attention decoder


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )
    def forward(self, x):
        return self.block(x)


class MultiScaleEdgeAttention(nn.Module):
    """Preserves fine vessels using multi-scale dilated context + learned edge attention."""
    def __init__(self, ch):
        super().__init__()
        mid = max(ch // 4, 16)
        self.d1 = nn.Conv2d(ch, mid, 3, padding=1, dilation=1, bias=False)
        self.d2 = nn.Conv2d(ch, mid, 3, padding=2, dilation=2, bias=False)
        self.d3 = nn.Conv2d(ch, mid, 3, padding=3, dilation=3, bias=False)
        self.mix = nn.Sequential(
            nn.Conv2d(mid * 3, ch, 1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU()
        )
        self.edge_gate = nn.Sequential(
            nn.Conv2d(ch, max(ch // 8, 8), 3, padding=1, bias=False),
            nn.GELU(),
            nn.Conv2d(max(ch // 8, 8), 1, 1),
            nn.Sigmoid()
        )
        self.out = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU()
        )

    def forward(self, x):
        z = torch.cat([self.d1(x), self.d2(x), self.d3(x)], dim=1)
        z = self.mix(z)
        gate = self.edge_gate(z)
        return self.out(x + z * gate)


class TransformerBottleneck(nn.Module):
    def __init__(self, in_ch=512, embed_dim=384, depth=4, heads=6, base_hw=16):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, 1)
        self.pos2d = nn.Parameter(torch.zeros(1, embed_dim, base_hw, base_hw))
        nn.init.trunc_normal_(self.pos2d, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.10, activation="gelu",
            batch_first=True, norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)
        self.out = nn.Conv2d(embed_dim, in_ch, 1)

    def forward(self, x):
        z = self.proj(x)
        B, C, H, W = z.shape
        pos = F.interpolate(self.pos2d, size=(H, W), mode="bilinear", align_corners=False)
        z = (z + pos).flatten(2).transpose(1, 2)
        z = self.enc(z).transpose(1, 2).reshape(B, C, H, W)
        return self.out(z)


class GridGraphLayer(nn.Module):
    """
    End-to-end graph message passing on the bottleneck feature grid.
    Each HxW feature location is a node; its 8 surrounding locations are neighbors.
    """
    def __init__(self, ch, dropout=0.10):
        super().__init__()
        self.self_proj = nn.Conv2d(ch, ch, 1, bias=False)
        self.neigh_proj = nn.Conv2d(ch, ch, 1, bias=False)
        self.norm = nn.BatchNorm2d(ch)
        self.drop = nn.Dropout2d(dropout)

    def forward(self, x):
        # 3x3 neighborhood = fixed spatial graph; remove the center to get neighbor messages.
        summed_3x3 = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1) * 9.0
        neigh = (summed_3x3 - x) / 8.0
        z = self.self_proj(x) + self.neigh_proj(neigh)
        z = self.drop(F.gelu(self.norm(z)))
        return x + z


class FeatureGNN(nn.Module):
    def __init__(self, ch=512, depth=4):
        super().__init__()
        self.layers = nn.ModuleList([GridGraphLayer(ch) for _ in range(depth)])
        self.out = nn.Sequential(
            nn.Conv2d(ch, ch, 1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU()
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x = self.out(x)
        # Kept for the existing STARE/HRF graph-node visualization cells.
        self.last_node_features = x.flatten(2).transpose(1, 2).detach()
        return x


class BranchFusion(nn.Module):
    """Learns how much Transformer and GNN information to use at each spatial location."""
    def __init__(self, ch):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(ch * 2, ch, 1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, 2, 1)
        )
        self.refine = DoubleConv(ch, ch)

    def forward(self, t, g):
        w = torch.softmax(self.gate(torch.cat([t, g], dim=1)), dim=1)
        z = w[:, 0:1] * t + w[:, 1:2] * g
        return self.refine(z)


class AttentionGate(nn.Module):
    def __init__(self, skip_ch, gate_ch, inter_ch):
        super().__init__()
        self.skip = nn.Conv2d(skip_ch, inter_ch, 1, bias=False)
        self.gate = nn.Conv2d(gate_ch, inter_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.GELU(),
            nn.Conv2d(inter_ch, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, skip, gate):
        a = self.psi(self.skip(skip) + self.gate(gate))
        return skip * a


class TransUNetGNNV6(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN encoder
        self.e1 = DoubleConv(3, 32)
        self.edge1 = MultiScaleEdgeAttention(32)
        self.e2 = DoubleConv(32, 64)
        self.edge2 = MultiScaleEdgeAttention(64)
        self.e3 = DoubleConv(64, 128)
        self.e4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.bridge = DoubleConv(256, 512)

        # Parallel global + structural reasoning
        self.trans = TransformerBottleneck(512, 384, depth=4, heads=6, base_hw=16)
        self.gnn = FeatureGNN(512, depth=4)
        self.fusion = BranchFusion(512)

        # Attention decoder
        self.u4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.a4 = AttentionGate(256, 256, 128)
        self.d4 = DoubleConv(512, 256)

        self.u3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.a3 = AttentionGate(128, 128, 64)
        self.d3 = DoubleConv(256, 128)

        self.u2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.a2 = AttentionGate(64, 64, 32)
        self.d2 = DoubleConv(128, 64)

        self.u1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.a1 = AttentionGate(32, 32, 16)
        self.d1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, 1, 1)
        self.aux2 = nn.Conv2d(64, 1, 1)
        self.aux3 = nn.Conv2d(128, 1, 1)

    def forward(self, x):
        e1 = self.edge1(self.e1(x))
        e2 = self.edge2(self.e2(self.pool(e1)))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))

        b = self.bridge(self.pool(e4))

        # Both branches see the SAME learned CNN features.
        t = self.trans(b)
        g = self.gnn(b)
        b = self.fusion(t, g)

        u4 = self.u4(b)
        d4 = self.d4(torch.cat([u4, self.a4(e4, u4)], dim=1))

        u3 = self.u3(d4)
        d3 = self.d3(torch.cat([u3, self.a3(e3, u3)], dim=1))

        u2 = self.u2(d3)
        d2 = self.d2(torch.cat([u2, self.a2(e2, u2)], dim=1))

        u1 = self.u1(d2)
        d1 = self.d1(torch.cat([u1, self.a1(e1, u1)], dim=1))

        main = self.out(d1)

        if self.training:
            aux2 = F.interpolate(self.aux2(d2), size=main.shape[-2:], mode="bilinear", align_corners=False)
            aux3 = F.interpolate(self.aux3(d3), size=main.shape[-2:], mode="bilinear", align_corners=False)
            return main, aux2, aux3

        return main


## 5. Hybrid V6 losses and metrics


In [ ]:
def soft_dice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    dims = (0,2,3)
    inter = (p*y).sum(dims)
    den = p.sum(dims) + y.sum(dims)
    return (1 - (2*inter+eps)/(den+eps)).mean()

def focal_tversky_loss(logits, y, alpha=0.55, beta=0.45, gamma=0.75, eps=1e-6):
    p = torch.sigmoid(logits)
    dims = (0,2,3)
    tp = (p*y).sum(dims)
    fp = (p*(1-y)).sum(dims)
    fn = ((1-p)*y).sum(dims)
    tv = (tp+eps)/(tp + alpha*fp + beta*fn + eps)
    return ((1-tv)**gamma).mean()

def soft_erode(img):
    p1 = -F.max_pool2d(-img, (3,1), (1,1), (1,0))
    p2 = -F.max_pool2d(-img, (1,3), (1,1), (0,1))
    return torch.minimum(p1, p2)

def soft_dilate(img):
    return F.max_pool2d(img, (3,3), (1,1), (1,1))

def soft_open(img):
    return soft_dilate(soft_erode(img))

def soft_skel(img, iters=10):
    img1 = soft_open(img)
    skel = F.relu(img-img1)
    for _ in range(iters):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img-img1)
        skel = skel + F.relu(delta-skel*delta)
    return skel

def soft_cldice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    sp = soft_skel(p, 10)
    sy = soft_skel(y, 10)

    tprec = (sp*y).sum((1,2,3))/(sp.sum((1,2,3))+eps)
    tsens = (sy*p).sum((1,2,3))/(sy.sum((1,2,3))+eps)
    cl = (2*tprec*tsens+eps)/(tprec+tsens+eps)
    return (1-cl).mean()

class V6Loss(nn.Module):
    def forward_one(self, logits, y):
        return (
            0.30*focal_tversky_loss(logits,y) +
            0.50*soft_dice_loss(logits,y) +
            0.20*soft_cldice_loss(logits,y)
        )

    def forward(self, outputs, y):
        if isinstance(outputs, (tuple,list)):
            main, a2, a3 = outputs
            return (
                self.forward_one(main,y) +
                0.25*self.forward_one(a2,y) +
                0.15*self.forward_one(a3,y)
            )
        return self.forward_one(outputs,y)

def binary_metrics(probs, targets, threshold):
    pred = (probs >= threshold).astype(np.uint8)
    gt = (targets > 0.5).astype(np.uint8)

    tp = np.logical_and(pred==1, gt==1).sum()
    fp = np.logical_and(pred==1, gt==0).sum()
    fn = np.logical_and(pred==0, gt==1).sum()

    dice = (2*tp+1e-6)/(2*tp+fp+fn+1e-6)
    precision = tp/(tp+fp+1e-6)
    recall = tp/(tp+fn+1e-6)
    return float(dice), float(precision), float(recall)

def cldice_np(pred, gt):
    sp = skeletonize(pred > 0)
    sg = skeletonize(gt > 0)

    tprec = (np.logical_and(sp,gt>0).sum()+1e-6)/(sp.sum()+1e-6)
    tsens = (np.logical_and(sg,pred>0).sum()+1e-6)/(sg.sum()+1e-6)
    return float((2*tprec*tsens+1e-6)/(tprec+tsens+1e-6))

def hd95_np(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    if not pred.any() or not gt.any():
        return float("nan")

    bp = np.logical_xor(
        pred,
        cv2.erode(pred.astype(np.uint8), np.ones((3,3),np.uint8)).astype(bool)
    )
    bg = np.logical_xor(
        gt,
        cv2.erode(gt.astype(np.uint8), np.ones((3,3),np.uint8)).astype(bool)
    )

    if not bp.any() or not bg.any():
        return float("nan")

    dtg = distance_transform_edt(~bg)
    dtp = distance_transform_edt(~bp)
    d = np.concatenate([dtg[bp], dtp[bg]])
    return float(np.percentile(d, 95))

def ssim_np(pred, gt):
    return float(structural_similarity(
        pred.astype(np.float32),
        gt.astype(np.float32),
        data_range=1.0
    ))


## 6. Fixed 16/4 STARE split — NO LOO


In [ ]:
IMAGE_SIZE = 512
PATCH_SIZE = 256
VAL_SPLIT = 0.20
BATCH_SIZE = 6
EPOCHS = 220
PATIENCE = 20
LR = 2e-4

# Scheduler choices: "plateau" or "onecycle".
# V5 defaults to ReduceLROnPlateau as requested.
# Never run both at the same time.
SCHEDULER_MODE = "plateau"

full_aug = StareDataset(STARE_ROOT,(IMAGE_SIZE,IMAGE_SIZE),augment=True)
full_clean = StareDataset(STARE_ROOT,(IMAGE_SIZE,IMAGE_SIZE),augment=False)

N = len(full_clean)
g = torch.Generator().manual_seed(SEED)
indices = torch.randperm(N, generator=g).tolist()

val_n = max(1, int(N*VAL_SPLIT))
val_idx = indices[:val_n]
train_idx = indices[val_n:]

train_full_aug = Subset(full_aug, train_idx)
train_clean = Subset(full_clean, train_idx)
val_clean = Subset(full_clean, val_idx)

train_patches = ThinVesselPatchDataset(
    train_full_aug,
    PATCH_SIZE,
    patches_per_image=32,
    min_vessel_fraction=0.006,
    max_vessel_fraction=0.12
)

train_loader = DataLoader(
    train_patches,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_clean,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

print(f"STARE: {len(train_idx)} train / {len(val_idx)} clean validation")
print("Train indices:", train_idx)
print("Validation indices:", val_idx)
print("Training patches/epoch:", len(train_patches))


## 7. TTA and threshold search


In [ ]:
@torch.no_grad()
def predict_tta(model, x):
    model.eval()
    preds = []

    transforms = [
        (lambda z:z, lambda z:z),
        (lambda z:torch.flip(z,[-1]), lambda z:torch.flip(z,[-1])),
        (lambda z:torch.flip(z,[-2]), lambda z:torch.flip(z,[-2])),
        (lambda z:torch.flip(z,[-2,-1]), lambda z:torch.flip(z,[-2,-1]))
    ]

    for fwd, inv in transforms:
        p = torch.sigmoid(model(fwd(x)))
        preds.append(inv(p))

    return torch.stack(preds).mean(0)

@torch.no_grad()
def collect_val_items(model, loader):
    items = []
    for x,y,f in loader:
        p = predict_tta(model, x.to(device))[0,0].cpu().numpy()
        gt = y[0,0].numpy()
        items.append((p,gt,f[0,0].numpy()))
    return items

def flatten_items(items):
    probs = np.concatenate([p[f>0.5].reshape(-1) for p,gt,f in items])
    targets = np.concatenate([gt[f>0.5].reshape(-1) for p,gt,f in items])
    return probs, targets

def find_best_threshold(items, lo=0.30, hi=0.90, step=0.01):
    best_t, best_d = 0.5, -1.0

    for t in np.arange(lo, hi+1e-9, step):
        scores = [
            binary_metrics(p[f>0.5].reshape(-1), gt[f>0.5].reshape(-1), float(t))[0]
            for p,gt,f in items
        ]
        score = float(np.mean(scores))

        if score > best_d:
            best_t, best_d = float(t), score

    return best_t, best_d


## 8. Callback-style utilities


In [ ]:
class ModelCheckpoint:
    def __init__(self, path):
        self.path = path
        self.best = -np.inf

    def step(self, score, model):
        if score > self.best + 1e-4:
            self.best = score
            torch.save(model.state_dict(), self.path)
            return True
        return False

class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -np.inf
        self.bad_epochs = 0

    def step(self, score):
        if score > self.best + self.min_delta:
            self.best = score
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

class CSVLogger:
    def __init__(self, path):
        self.path = path
        self.header_written = False

    def log(self, row):
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if not self.header_written and os.path.getsize(self.path) == 0:
                writer.writeheader()
                self.header_written = True
            writer.writerow(row)

class BestMetricTracker:
    def __init__(self):
        self.best_dice = -np.inf
        self.best_epoch = 0
        self.best_threshold = 0.5

    def update(self, epoch, dice, threshold):
        if dice > self.best_dice:
            self.best_dice = dice
            self.best_epoch = epoch
            self.best_threshold = threshold

class GradientNormMonitor:
    @staticmethod
    def value(model):
        total = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total += p.grad.detach().norm(2).item() ** 2
        return total ** 0.5

def invalid_loss(loss):
    return not torch.isfinite(loss).item()


## 9. Train Hybrid TransUNet-GNN V6

### Learning-rate behavior
Default: **ReduceLROnPlateau** watches validation Dice. If validation Dice stops improving for 5 epochs, LR is multiplied by 0.5, down to `1e-6`.

`OneCycleLR` is also available by changing `SCHEDULER_MODE = "onecycle"`, but the notebook never uses OneCycleLR and ReduceLROnPlateau simultaneously.


In [ ]:
TRANS_CHECKPOINT='stare_hybrid_v6_best.pt'; TRANS_THRESHOLD_FILE='stare_hybrid_v6_threshold.txt'; TRANS_HISTORY_FILE='stare_hybrid_v6_history.csv'; open(TRANS_HISTORY_FILE,'w').close()
model=TransUNetGNNV6().to(device); loss_fn=V6Loss().to(device); optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=5,threshold=1e-4,min_lr=1e-6)
checkpoint=ModelCheckpoint(TRANS_CHECKPOINT); early_stop=EarlyStopping(PATIENCE); logger=CSVLogger(TRANS_HISTORY_FILE); tracker=BestMetricTracker(); grad_monitor=GradientNormMonitor(); history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); running=0.; last_grad_norm=np.nan
    for x,y,f in train_loader:
        x,y,f=x.to(device,non_blocking=True),y.to(device,non_blocking=True),f.to(device,non_blocking=True); y=y*f; optimizer.zero_grad(set_to_none=True); outputs=model(x); loss=loss_fn(outputs,y)
        if invalid_loss(loss): raise RuntimeError('NaN/Inf loss detected')
        loss.backward(); last_grad_norm=grad_monitor.value(model); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step(); running+=loss.item()*x.size(0)
    train_loss=running/len(train_loader.dataset); val_items=collect_val_items(model,val_loader); threshold,val_dice=find_best_threshold(val_items); scheduler.step(val_dice); lr_now=optimizer.param_groups[0]['lr']; improved=checkpoint.step(val_dice,model); tracker.update(epoch,val_dice,threshold)
    if improved: open(TRANS_THRESHOLD_FILE,'w').write(str(threshold))
    row={'epoch':epoch,'train_loss':train_loss,'val_dice':val_dice,'best_dice':tracker.best_dice,'threshold':threshold,'lr':lr_now,'grad_norm':last_grad_norm}; history.append(row); logger.log(row)
    print(f'Epoch {epoch:3d}/{EPOCHS} | loss={train_loss:.4f} | val_dice={val_dice:.4f} | best_dice={tracker.best_dice:.4f} | thr={threshold:.2f} | lr={lr_now:.2e}')
    if improved: print(f'NEW BEST DICE: {val_dice:.4f}')
    if early_stop.step(val_dice): print('Early stopping at epoch',epoch); break
model.load_state_dict(torch.load(TRANS_CHECKPOINT,map_location=device)); best_threshold=float(open(TRANS_THRESHOLD_FILE).read()); model.eval(); print('Main-stage best Dice:',tracker.best_dice,'threshold:',best_threshold)


## 10. Final Hybrid V6 metrics — Dice, SSIM, HD95, ROC-AUC and PR-AUC


In [ ]:
model.load_state_dict(torch.load(TRANS_CHECKPOINT, map_location=device))
model.eval()

best_threshold = float(open(TRANS_THRESHOLD_FILE).read())
val_items = collect_val_items(model, val_loader)
val_probs, val_targets = flatten_items(val_items)

dice, precision, recall = binary_metrics(val_probs, val_targets, best_threshold)
roc_auc = roc_auc_score(val_targets, val_probs)
pr_auc = average_precision_score(val_targets, val_probs)

per_dice = []
per_cldice = []
per_hd95 = []
per_ssim = []

for p,gt,f in val_items:
    valid=f>0.5
    pred = ((p >= best_threshold) & valid).astype(np.uint8)
    gt_eval=((gt>0.5) & valid).astype(np.uint8)

    per_dice.append(binary_metrics(
        p[valid].reshape(-1), gt[valid].reshape(-1), best_threshold
    )[0])
    per_cldice.append(cldice_np(pred,gt_eval))
    per_hd95.append(hd95_np(pred,gt_eval))
    per_ssim.append(ssim_np(pred,gt_eval))

print("STARE TransUNet+GNN V6 clean validation metrics")
print(f"Dice      : {np.mean(per_dice):.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC/AP : {pr_auc:.4f}")
print(f"clDice    : {np.nanmean(per_cldice):.4f}")
print(f"HD95      : {np.nanmean(per_hd95):.4f} pixels")
print(f"SSIM      : {np.mean(per_ssim):.4f}")
print(f"Threshold : {best_threshold:.2f}")


## 11. ROC curve — area under curve


In [ ]:
fpr, tpr, _ = roc_curve(val_targets, val_probs)

plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],"--",label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("STARE TransUNet+GNN V6 ROC Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 12. Precision–Recall curve — PR area under curve


In [ ]:
pr_precision, pr_recall, _ = precision_recall_curve(val_targets, val_probs)

plt.figure(figsize=(7,6))
plt.plot(pr_recall, pr_precision, label=f"PR-AUC/AP = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("STARE TransUNet+GNN V6 Precision–Recall Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 13. Confusion matrix


In [ ]:
val_pred = (val_probs >= best_threshold).astype(np.uint8)
cm = confusion_matrix(val_targets.astype(np.uint8), val_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("TransUNet+GNN V6 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0,1],["Background","Vessel"])
plt.yticks([0,1],["Background","Vessel"])

for i in range(2):
    for j in range(2):
        plt.text(j,i,f"{cm[i,j]:,}",ha="center",va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


## 14. Training curves: loss, validation Dice, learning rate and gradient norm


In [ ]:
hist = pd.DataFrame(history)

for column, ylabel, title in [
    ("train_loss","Loss","Training Loss"),
    ("val_dice","Dice","Validation Dice"),
    ("lr","Learning rate","Learning Rate"),
    ("grad_norm","Gradient norm","Gradient Norm")
]:
    plt.figure(figsize=(8,5))
    plt.plot(hist["epoch"], hist[column])
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.show()


## 15. Threshold-vs-Dice curve


In [ ]:
thresholds = np.arange(0.20,0.901,0.01)
threshold_scores = []

for t in thresholds:
    scores = [
        binary_metrics(p[f>0.5].reshape(-1),gt[f>0.5].reshape(-1),float(t))[0]
        for p,gt,f in val_items
    ]
    threshold_scores.append(np.mean(scores))

plt.figure(figsize=(8,5))
plt.plot(thresholds, threshold_scores)
plt.axvline(best_threshold, linestyle="--", label=f"Best threshold={best_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Mean validation Dice")
plt.title("Threshold vs Dice")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 16. Visual metric comparison


In [ ]:
metric_names = ["Dice","Precision","Recall","ROC-AUC","PR-AUC","clDice","SSIM"]
metric_values = [
    np.mean(per_dice), precision, recall, roc_auc,
    pr_auc, np.nanmean(per_cldice), np.mean(per_ssim)
]

plt.figure(figsize=(10,5))
plt.bar(metric_names, metric_values)
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("STARE TransUNet+GNN V6 Validation Metrics")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## ALL 20 STARE predictions — 16 training + 4 validation


In [ ]:
@torch.no_grad()
def show_all_20_predictions():
    model.eval()
    for i,(x,y,f) in enumerate(full_clean):
        p=predict_tta(model,x.unsqueeze(0).to(device))[0,0].cpu().numpy()
        gt=y[0].numpy(); fv=f[0].numpy()>0.5
        pred=((p>=best_threshold)&fv).astype(np.uint8)
        gt_eval=((gt>0.5)&fv).astype(np.uint8)
        d=binary_metrics(p[fv],gt[fv],best_threshold)[0]
        s=ssim_np(pred,gt_eval); h=hd95_np(pred,gt_eval)
        med=float(np.median(p[fv]))
        img=np.clip(x.permute(1,2,0).numpy()*.5+.5,0,1)
        split='Validation' if i in val_idx else 'Training'
        err=np.zeros((*gt.shape,3),dtype=float)
        err[:,:,0]=np.logical_and(pred==1,gt_eval==0)
        err[:,:,1]=np.logical_and(pred==1,gt_eval==1)
        err[:,:,2]=np.logical_and(pred==0,gt_eval==1)
        fig,ax=plt.subplots(1,5,figsize=(25,5))
        ax[0].imshow(img); ax[0].set_title(f'STARE {i+1} — {split}')
        ax[1].imshow(gt_eval,cmap='gray'); ax[1].set_title('Ground Truth')
        ax[2].imshow(p,cmap='gray',vmin=0,vmax=1); ax[2].set_title(f'Probability\nMedian={med:.4f}')
        ax[3].imshow(pred,cmap='gray'); ax[3].set_title(f'Prediction\nDice={d:.4f} SSIM={s:.4f} HD95={h:.2f}')
        ax[4].imshow(err); ax[4].set_title('TP green / FP red / FN blue')
        for a in ax:a.axis('off')
        plt.tight_layout(); plt.show()
show_all_20_predictions()


## GNN graph/node visualizations for all 20 images

This uses **GCN-style GNN message passing**, not GNN. It visualizes the bottleneck graph nodes and graph structure used during end-to-end segmentation.


In [ ]:
def show_gnn_graph(image_index, stride=2):

    model.eval()

    # STARE dataset
    x, y, f = full_clean[image_index]

    xb = x.unsqueeze(0).to(device)

    with torch.no_grad():
        _ = model(xb)

    node_features = model.gnn.last_node_features

    if node_features is None:
        raise RuntimeError(
            "GNN node features were not stored."
        )

    node_features = node_features.detach().cpu()

    print(
        f"Image {image_index + 1} | "
        f"GNN feature shape: {tuple(node_features.shape)}"
    )

    # --------------------------------------------------
    # Convert GNN features to node strength
    # --------------------------------------------------

    if node_features.ndim == 4:
        # [B, C, H, W]
        features = node_features[0]

        strength = torch.norm(
            features,
            dim=0
        ).numpy()

        Hn, Wn = strength.shape

    elif node_features.ndim == 3:

        if node_features.shape[0] == 1:
            # [B, N, C]

            features = node_features[0]

            strength = torch.norm(
                features,
                dim=-1
            ).numpy()

            N = strength.shape[0]

            Hn = int(np.sqrt(N))

            while Hn > 1 and N % Hn != 0:
                Hn -= 1

            Wn = N // Hn

            strength = strength.reshape(Hn, Wn)

        else:
            # [C, H, W]

            strength = torch.norm(
                node_features,
                dim=0
            ).numpy()

            Hn, Wn = strength.shape

    elif node_features.ndim == 2:
        # [N, C]

        strength = torch.norm(
            node_features,
            dim=-1
        ).numpy()

        N = strength.shape[0]

        Hn = int(np.sqrt(N))

        while Hn > 1 and N % Hn != 0:
            Hn -= 1

        Wn = N // Hn

        strength = strength.reshape(Hn, Wn)

    else:
        raise ValueError(
            f"Unexpected GNN feature shape: "
            f"{tuple(node_features.shape)}"
        )

    # Normalize
    strength = strength.astype(np.float32)

    strength = (
        strength - strength.min()
    ) / (
        strength.max() - strength.min() + 1e-8
    )

    # Original image
    img = x.permute(1, 2, 0).numpy()

    img = np.clip(
        img * 0.5 + 0.5,
        0,
        1
    )

    # Graph coordinates
    yy = np.arange(0, Hn, stride)
    xx = np.arange(0, Wn, stride)

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(14, 6)
    )

    ax[0].imshow(img)

    ax[0].set_title(
        f"STARE Image {image_index + 1}"
    )

    ax[0].axis("off")

    ax[1].imshow(
        strength,
        cmap="hot",
        interpolation="nearest"
    )

    # Draw graph
    for y_pos in yy:

        for x_pos in xx:

            # node
            ax[1].scatter(
                x_pos,
                y_pos,
                s=10,
                c="cyan"
            )

            # horizontal
            if x_pos + stride < Wn:
                ax[1].plot(
                    [x_pos, x_pos + stride],
                    [y_pos, y_pos],
                    color="white",
                    linewidth=0.5,
                    alpha=0.5
                )

            # vertical
            if y_pos + stride < Hn:
                ax[1].plot(
                    [x_pos, x_pos],
                    [y_pos, y_pos + stride],
                    color="white",
                    linewidth=0.5,
                    alpha=0.5
                )

            # diagonal
            if (
                x_pos + stride < Wn
                and y_pos + stride < Hn
            ):
                ax[1].plot(
                    [x_pos, x_pos + stride],
                    [y_pos, y_pos + stride],
                    color="white",
                    linewidth=0.35,
                    alpha=0.3
                )

    ax[1].set_title(
        f"GNN Node Feature Strength\n"
        f"Graph Size = {Hn} x {Wn}"
    )

    ax[1].axis("off")

    plt.tight_layout()
    plt.show()


# ==========================================================
# SHOW ALL 20 STARE IMAGES
# ==========================================================

for i in range(20):

    print("\n" + "=" * 60)

    print(
        f"GNN GRAPH VISUALIZATION {i + 1}/20"
    )

    print("=" * 60)

    show_gnn_graph(
        image_index=i,
        stride=2
    )

## Download V6 checkpoints


In [ ]:
@torch.no_grad()
def medians_law_report():
    rows=[]
    model.eval()
    for i,(x,y,f) in enumerate(full_clean):
        p=predict_tta(model,x.unsqueeze(0).to(device))[0,0].cpu().numpy()
        gt=y[0].numpy(); fv=f[0].numpy()>0.5
        pred=(p>=best_threshold).astype(np.uint8)
        d=binary_metrics(p[fv],gt[fv],best_threshold)[0]
        ss=structural_similarity((pred*fv).astype(float),((gt>0.5)*fv).astype(float),data_range=1.0)
        hd=hd95_np(pred*fv,(gt>0.5).astype(np.uint8)*fv)
        rows.append({
            "Image":i+1,
            "Split":"Validation" if i in val_idx else "Training",
            "Median_Probability_FOV":float(np.median(p[fv])),
            "Dice":d,"SSIM":ss,"HD95_px":hd
        })
    df=pd.DataFrame(rows)
    display(df)
    print("Median of the 20 per-image median probabilities:",
          f"{df['Median_Probability_FOV'].median():.6f}")
    return df

median_df=medians_law_report()


In [ ]:
from google.colab import files
for p in [TRANS_CHECKPOINT,TRANS_THRESHOLD_FILE,TRANS_HISTORY_FILE,'stare_v6_finetuned_best.pt','stare_v6_finetuned_threshold.txt']:
    if os.path.exists(p): files.download(p)


## Reporting note
All 20 predictions are displayed as requested. **Only the 4 held-out validation images should be used as the reported validation score**; the other 16 were seen during training. The GNN graph maps are diagnostic visualizations, not segmentation metrics.


## Hybrid V6 architecture check


In [ ]:
print("Dataset: STARE")
print("Architecture: CNN + multi-scale edge enhancement + parallel Transformer/Feature-GNN + learned fusion + attention decoder")
print("GNN depth: 4")
print("Loss: 50% Dice + 30% Focal-Tversky + 20% clDice")
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
